### En este proyecto, se proporciona un conjunto de datos de anuncios de venta de coches.

#### Análisis: Carga y Exploración Inicial de Datos

En este primer paso del proyecto, realizamos la preparación de nuestro entorno de trabajo y una exploración inicial de la estructura del conjunto de datos:

1. **Importación de Librerías**:
   * `pandas` (como `pd`): La herramienta principal en Python para la manipulación y análisis de datos tabulares a través de estructuras llamadas *DataFrames*.
   * `plotly.express` (como `px`): Una librería de visualización interactiva que utilizaremos más adelante para crear gráficos dinámicos y atractivos.

2. **Lectura del Archivo CSV**:
   * Cargamos el archivo `vehicles_us.csv` utilizando el método `pd.read_csv()`, almacenando la información en la variable `df`.

3. **Visualización de las Primeras Filas**:
   * Con el método `df.head()`, imprimimos de forma tabular los primeros 5 registros de la tabla para entender visualmente qué información contiene cada columna (como el precio, año del modelo, odómetro, transmisión, etc.).

4. **Inspección de la Estructura Técnica (`df.info()`)**:
   * Obtenemos un resumen que nos indica el número total de filas (51,525 registros) y columnas (13 variables).
   * Identificamos de manera inmediata qué columnas contienen **valores ausentes (nulos)** al comparar el recuento de datos no nulos con el total (por ejemplo, las columnas `model_year`, `cylinders`, `odometer`, `paint_color` e `is_4wd` presentan valores faltantes que requerirán depuración).
   * Evaluamos los tipos de datos asignados automáticamente (enteros, decimales y objetos de texto), lo que nos ayuda a planificar las conversiones de tipo necesarias para un análisis riguroso.

In [19]:
import pandas as pd
import plotly.express as px

# 1. Cargar el conjunto de datos de vehículos
df = pd.read_csv('../vehicles_us.csv')

# Mostrar las primeras filas y estructura básica
print("Primeras filas del conjunto de datos de vehículos:")
display(df.head())
print("\nInformación general del dataset:")
print(df.info())

Primeras filas del conjunto de datos de vehículos:


,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28



Información general del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB
None


#### Limpieza y Depuración de los Datos 

Con base en el diagnóstico anterior, aplicamos las siguientes estrategias de limpieza en el bloque de código de arriba para asegurar la consistencia del análisis:

Tratamiento de is_4wd: Asumimos que la ausencia de valor (nulo) en la columna de tracción en las cuatro ruedas equivale a que el vehículo no la posee. Por lo tanto, rellenamos estos vacíos con 0 (Falso/No) y posteriormente convertimos la columna a tipo entero (int).

Tratamiento de paint_color: Para no perder registros valiosos por la falta de información del color de la pintura, imputamos los valores nulos con la etiqueta 'unknown' (desconocido).

Imputación Inteligente de model_year, cylinders y odometer: En lugar de rellenar los datos de estas tres columnas con una mediana general (que sesgaría los datos, por ejemplo, asignando el mismo kilometraje a un camión pesado que a un auto compacto), agrupamos los datos por el modelo del vehículo (model) y calculamos la mediana específica de ese modelo para rellenar sus nulos correspondientes.

* `Ejemplo:` Un vacío en el año de un 'Ford F-150' se llena con la mediana de los años de los 'Ford F-150' existentes.

* `Mediana Residual:` En caso de que un modelo en particular tuviera únicamente valores nulos en su grupo (evitando que se calcule una mediana grupal), aplicamos como respaldo la mediana de todo el conjunto de datos.

* `Conversión de Tipos de Datos:` Finalmente, transformamos model_year, cylinders e is_4wd a tipo entero (int), ya que conceptualmente no pueden tener decimales (un vehículo no puede ser del año 2011.5 ni tener 5.5 cilindros). Con esto, el dataset queda 100% limpio y listo para el análisis gráfico sin pérdida de registros.

In [20]:
# 1. Rellenar 'is_4wd' con 0 (asumiendo que los valores nulos significan que no es 4WD)
df['is_4wd'] = df['is_4wd'].fillna(0)

# 2. Rellenar 'paint_color' con 'unknown'
df['paint_color'] = df['paint_color'].fillna('unknown')

# 3. Imputar 'model_year', 'cylinders' y 'odometer' usando la mediana agrupada por 'model'
for col in ['model_year', 'cylinders', 'odometer']:
    df[col] = df.groupby('model')[col].transform(lambda x: x.fillna(x.median()))

# Si quedan valores nulos residuales (modelos con solo valores nulos en esas columnas), usamos la mediana general
for col in ['model_year', 'cylinders', 'odometer']:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# 4. Asegurar tipos de datos correctos
df['model_year'] = df['model_year'].astype(int)
df['cylinders'] = df['cylinders'].astype(int)
df['is_4wd'] = df['is_4wd'].astype(int)

# 5. Mostrar resultados de la imputación
print("Valores ausentes restantes por columna:")
display(df.isnull().sum().to_frame(name='valores_ausentes'))

print("\nPrimeras filas del DataFrame limpio:")
display(df.head())
print("\nInformación general del dataset:")
print(df.info())


Valores ausentes restantes por columna:


,valores_ausentes
price,0
model_year,0
model,0
condition,0
cylinders,0
fuel,0
odometer,0
transmission,0
type,0
paint_color,0



Primeras filas del DataFrame limpio:


,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011,bmw x5,good,6,gas,145000.0,automatic,SUV,unknown,1,2018-06-23,19
1,25500,2011,ford f-150,good,6,gas,88705.0,automatic,pickup,white,1,2018-10-19,50
2,5500,2013,hyundai sonata,like new,4,gas,110000.0,automatic,sedan,red,0,2019-02-07,79
3,1500,2003,ford f-150,fair,8,gas,121928.0,automatic,pickup,unknown,0,2019-03-22,9
4,14900,2017,chrysler 200,excellent,4,gas,80903.0,automatic,sedan,black,0,2019-04-02,28



Información general del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    51525 non-null  int64  
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     51525 non-null  int64  
 5   fuel          51525 non-null  str    
 6   odometer      51525 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   51525 non-null  str    
 10  is_4wd        51525 non-null  int64  
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(1), int64(5), str(7)
memory usage: 7.7 MB
None


Se realizan cuatro gráficos interactivos utilizando **Plotly Express** a partir de nuestro conjunto de datos depurado. A continuación se detalla qué analiza cada uno:

1. **Distribución de los Vehículos por Año de Modelo**:
   * **Qué muestra**: Un histograma de la variable `model_year`.
   * **Análisis**: Al haber eliminado los valores nulos, ahora podemos ver la distribución completa y real del año de fabricación de los vehículos. Se observa de manera clara un sesgo a la izquierda: la gran mayoría de los anuncios corresponden a vehículos fabricados a partir del año 2000, con un pico de concentración en la década de 2010. Esto nos indica que el mercado de coches usados analizado es mayoritariamente moderno.

2. **Relación entre Año del Modelo y Precio (Coloreado por Transmisión)**:
   * **Qué muestra**: Un gráfico de dispersión utilizando una muestra aleatoria de 5,000 registros para evitar la superposición extrema de puntos.
   * **Análisis**: Permite evaluar de forma visual la depreciación de los vehículos. Observamos una tendencia clara donde los autos más recientes (años cercanos a 2018-2019) alcanzan precios máximos significativamente más altos que los vehículos antiguos. Además, el código colorea los puntos según el tipo de transmisión (`transmission`), lo que nos facilita identificar visualmente si las transmisiones automáticas, manuales u otras se concentran en ciertos rangos de precio o antigüedad.

3. **Distribución de Precios: Tracción Simple vs 4WD (Percentil 95)**:
   * **Qué muestra**: Un gráfico de cajas (*boxplot*) que compara la distribución de los precios de venta agrupados según el tipo de tracción del vehículo (`is_4wd`), limitando el análisis al percentil 95 para ignorar valores atípicos extremos.
   * **Análisis**: Los gráficos de caja son excelentes para comparar distribuciones. Al mapear previamente la columna numérica `is_4wd` a etiquetas de texto descriptivas ('Tracción Simple' y 'Tracción 4WD'), la visualización muestra con claridad la diferencia entre ambos grupos. Generalmente, se puede evaluar si los vehículos con tracción integral (4WD) tienen una mediana de precio más alta y un rango intercuartílico desplazado hacia arriba en comparación con los de tracción simple.

4. **Matriz de Correlación Completa**:
   * **Qué muestra**: Un mapa de calor interactivo (`imshow`) que calcula el coeficiente de correlación de Pearson entre todas las variables numéricas.
   * **Análisis**: Dado que ya no existen valores nulos (`NaN`) en columnas como `model_year`, `cylinders` u `odometer`, la matriz de correlación se calcula sobre el 100% de los datos sin sesgos de registros faltantes. El mapa de calor revela la fuerza y dirección de las relaciones:
     * Por ejemplo, esperamos una **correlación negativa** notable entre el kilometraje (`odometer`) y el precio (`price`), lo que confirma que a mayor kilometraje, menor es el valor del vehículo.
     * De igual forma, podemos evaluar correlaciones positivas lógicas (como la relación entre el número de cilindros `cylinders` y el tamaño/tipo de motor del coche).

In [21]:
import plotly.express as px

# 1. Distribución del año de los modelos (ahora sin nulos y como enteros)
fig_years = px.histogram(
    df,
    x='model_year',
    title='Distribución de los Vehículos por Año de Modelo (Datos Limpios)',
    labels={'model_year': 'Año del Modelo'},
    color_discrete_sequence=['#2ca02c'],
    nbins=30
)
fig_years.update_layout(bargap=0.1)
fig_years.show()

# 2. Relación entre Año del Modelo y Precio, coloreado por tipo de transmisión
fig_scatter_clean = px.scatter(
    df.sample(n=min(5000, len(df)), random_state=42),
    x='model_year',
    y='price',
    color='transmission',
    title='Precio vs Año del Modelo (Muestra de 5000 vehículos limpios)',
    labels={'model_year': 'Año del Modelo', 'price': 'Precio ($)'},
    opacity=0.6
)
fig_scatter_clean.show()

# 3. Comparación del Precio según si es Tracción 4x4 (is_4wd) usando un gráfico de caja
# Convertimos temporalmente is_4wd a texto para una mejor leyenda
df_temp = df.copy()
df_temp['is_4wd'] = df_temp['is_4wd'].map({1: 'Tracción 4WD', 0: 'Tracción Simple'})

fig_box_4wd = px.box(
    df_temp[df_temp['price'] < df_temp['price'].quantile(0.95)],
    x='is_4wd',
    y='price',
    color='is_4wd',
    title='Distribución de Precios: Tracción Simple vs 4WD (Percentil 95)',
    labels={'is_4wd': 'Tipo de Tracción', 'price': 'Precio ($)'}
)
fig_box_4wd.show()

# 4. Nueva Matriz de Correlación interactiva con todas las variables numéricas completas
corr_matrix_clean = df.select_dtypes(include=['float64', 'int64']).corr()
fig_corr_clean = px.imshow(
    corr_matrix_clean,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='Viridis',
    title='Matriz de Correlación Completa (Sin valores nulos)'
)
fig_corr_clean.show()

Visualizamos dos gráficos interactivos muy importantes para comprender el valor comercial de los vehículos utilizando los datos limpios:

1. **Distribución de Precios de Vehículos (Percentil 99)**:
   * **Qué muestra**: Un histograma de la variable `price` (precio), excluyendo el 1% de los datos con los precios más altos (valores atípicos extremos como vehículos de colección o errores de captura).
   * **Análisis**: La gráfica muestra una distribución sesgada a la derecha (*asimetría positiva*). El pico más alto de publicaciones se concentra en el rango de precios bajos y medios (aproximadamente entre $3,000 y $12,000). A medida que el precio aumenta, la cantidad de anuncios disminuye paulatinamente. Esto refleja un mercado de vehículos usados altamente accesible, donde la gran mayoría de la oferta se concentra en opciones económicas.

2. **Relación entre Odómetro (Kilometraje) y Precio (Muestra de 5,000 registros)**:
   * **Qué muestra**: Un gráfico de dispersión que compara el kilometraje acumulado (`odometer`) en el eje horizontal contra el precio de venta (`price`) en el eje vertical, coloreando los puntos según la condición estética y mecánica del vehículo (`condition`).
   * **Análisis**:
     * **Relación Inversa (Tendencia)**: Se observa claramente que a medida que el kilometraje aumenta (se desplaza a la derecha en el eje X), el precio de los vehículos tiende a bajar considerablemente. Esto confirma la teoría de la depreciación por uso.
     * **Impacto de la Condición**: El color de los puntos nos permite evaluar cómo influye el estado del auto. Por ejemplo, los vehículos etiquetados en condiciones como `'excellent'` (excelente) o `'like new'` (como nuevo) suelen ubicarse en la parte superior del gráfico (precios más altos) para un mismo nivel de kilometraje, mientras que los vehículos en condición `'fair'` (aceptable) o `'salvage'` (recuperado) se concentran en la zona inferior de precios.

In [22]:
# 1. Histograma de Precios Actualizado (Datos Limpios)
fig_hist_clean = px.histogram(
    df[df['price'] < df['price'].quantile(0.99)],
    x='price',
    nbins=50,
    title='Distribución de Precios de Vehículos (Datos Limpios - Percentil 99)',
    labels={'price': 'Precio ($)'},
    color_discrete_sequence=['#1f77b4']
)
fig_hist_clean.update_layout(bargap=0.1)
fig_hist_clean.show()

# 2. Gráfico de Dispersión Actualizado: Odómetro vs Precio (Muestra de 5000 registros, datos limpios e imputados)
fig_scatter_clean_odo = px.scatter(
    df.sample(n=min(5000, len(df)), random_state=42),
    x='odometer',
    y='price',
    color='condition',
    title='Relación entre Odómetro y Precio (Datos Limpios e Imputados - Muestra de 5000)',
    labels={'odometer': 'Odómetro (Millas/Km)', 'price': 'Precio ($)', 'condition': 'Condición'},
    opacity=0.6
)
fig_scatter_clean_odo.show()

### Conclusión General

Con base en las variables analizadas y las relaciones estadísticas observadas en el conjunto de datos, se extraen las siguientes conclusiones sobre el mercado de coches usados:

1. **Comportamiento del Valor y Depreciación**:
   * **Impacto del Kilometraje**: Se confirma una relación inversa clara entre el kilometraje (`odometer`) y el precio del vehículo. El valor comercial disminuye de forma constante a medida que aumenta el uso acumulado del automóvil.
   * **Influencia de la Antigüedad**: Existe una tendencia natural de depreciación donde los modelos más recientes concentran los precios de lista más altos, mientras que los vehículos de años anteriores se deprecian progresivamente.

2. **Estructura y Perfil del Catálogo**:
   * **Segmento de Precios**: El mercado analizado es predominantemente de gama media-baja o accesible, concentrando la gran mayoría de la oferta de anuncios en un rango de precios que oscila entre los $3,000 y $12,000 dólares.
   * **Modernidad de la Oferta**: La disponibilidad de vehículos está fuertemente inclinada hacia modelos recientes fabricados a partir del año 2000, con una densidad máxima en la década de 2010.

3. **Factores Clave en el Precio de Venta**:
   * **Tipo de Tracción**: La presencia de tracción en las cuatro ruedas (`is_4wd`) actúa como un factor de valor agregado. Los vehículos con tracción 4WD registran una distribución de precios con medianas y máximos superiores en comparación con los vehículos de tracción simple.
   * **Estado del Vehículo**: La condición estética y mecánica del automóvil (`condition`) influye directamente en la retención del valor, permitiendo que vehículos con un kilometraje similar se coticen a un precio sustancialmente mayor si se encuentran en excelente estado.